# Scene Feasibility Review

Iterative manual review of cached BridgeData V2 scenes for contrastive-pair
feasibility. Each session loads the review queue from `manifest.csv`, shows
the initial frame (and grasp frame when present) with both sides of the
antonym swap, and writes labels one scene at a time via
`update_manifest_annotations`. Labels live only in the Drive manifest; this
notebook does not hold an annotations dict.

Primary-stratum scenes require `feasible_both == 'yes'`. Values `no` and
`unclear` stay out of that stratum. The model and the TFDS stream are never
loaded here.

**Prerequisite:** Notebook 02 has cached frames and a manifest under
`openvla_cache/bridge_multiobj/`.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/bridge_multiobj'
assert os.path.isfile(os.path.join(CACHE_DIR, 'manifest.csv')), (
    f'No manifest at {CACHE_DIR}; run Notebook 02 cache_records first.'
)
print('cache ->', CACHE_DIR)

## 2. Import `data.py`

Imports review and annotation helpers from the Drive-synced repository. Mount
Drive in section 1 first. The default path is
`/content/drive/Othercomputers/My MacBook Pro/ECS8056`; change `REPO_DIR` in
the next cell if the folder lives elsewhere on Drive.

In [ ]:
import sys, os, importlib

# Drive-synced repository (mount Drive first). Change if the folder path differs.
REPO_DIR = '/content/drive/Othercomputers/My MacBook Pro/ECS8056'


def find_repo_dir(anchor):
    """Return the directory holding `anchor` from known Drive paths only."""
    candidates = []
    if REPO_DIR:
        candidates.append(REPO_DIR)
    candidates.extend([
        '/content/drive/Othercomputers/My MacBook Pro/ECS8056',
        '/content/drive/MyDrive/ECS8056',
        '/content/ECS8056',
    ])
    seen = set()
    for d in candidates:
        d = os.path.abspath(d)
        if d in seen:
            continue
        seen.add(d)
        if os.path.isfile(os.path.join(d, anchor)):
            return d
    return None


module_dir = find_repo_dir('data.py')
if module_dir is None:
    raise FileNotFoundError(
        f"data.py not found on Google Drive. Mount Drive in section 1, confirm ECS8056 has finished syncing, then set REPO_DIR to the folder that contains data.py (tried REPO_DIR={REPO_DIR!r}). Also check with:\n  !ls /content/drive/Othercomputers/My MacBook Pro/ECS8056")
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
sys.modules.pop('data', None)
importlib.invalidate_caches()

from data import (
    CATEGORY_OTHER,
    CATEGORY_PLACEMENT,
    CATEGORY_REFERENT,
    review_queue,
    review_summary,
    update_manifest_annotations,
)
print('imported data.py from', module_dir)

## 3. Review progress

Pairable `referent_selection` scenes still marked `unreviewed` are the main
headroom for growing the primary stratum. Re-run this cell after a labeling
session to confirm the counts moved.

In [ ]:
summary = review_summary(CACHE_DIR)
print(f"total scenes: {summary['total']}")
print(f"pairable: {summary['pairable']}  non-pairable: {summary['non_pairable']}")
print(
    f"referent_selection pairable: "
    f"unreviewed={summary['referent_pairable_unreviewed']}  "
    f"yes={summary['referent_pairable_yes']}"
)
print('\nby category x feasible_both:')
for (cat, feas), n in sorted(summary['by_category_feasibility'].items()):
    print(f'  {cat:22} {feas:12} {n}')

## 4. Interactive reviewer

Judge whether the implied placement or referent choice is physically possible
on **both** sides of the antonym swap. Default queue: `unreviewed` pairable
scenes, ordered so `referent_selection` comes first. Change the status filter
to revisit `yes`, `no`, or `unclear`.

Each save writes a single row to the Drive manifest and advances. Skip leaves
the row unchanged.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display

# --- Session filters (edit and re-run this cell to change the queue) ---------
STATUS = 'unreviewed'   # or 'yes' / 'no' / 'unclear' / None for all
CATEGORIES = None       # e.g. [CATEGORY_REFERENT] to restrict
ONLY_PAIRABLE = True
BATCH_SIZE = 40         # max scenes loaded for this session

queue = review_queue(
    CACHE_DIR,
    status=STATUS,
    categories=CATEGORIES,
    only_pairable=ONLY_PAIRABLE,
    limit=BATCH_SIZE,
)
print(f'queue length: {len(queue)} (status={STATUS!r}, limit={BATCH_SIZE})')
if not queue:
    print('Nothing to review under the current filters.')

state = {'idx': 0, 'done': 0}

status_label = widgets.HTML()
meta_label = widgets.HTML()
img_out = widgets.Output()
note_box = widgets.Text(
    description='Note',
    placeholder='optional feasibility note',
    layout=widgets.Layout(width='90%'),
)
cat_dd = widgets.Dropdown(
    options=[
        ('(keep heuristic)', ''),
        (CATEGORY_REFERENT, CATEGORY_REFERENT),
        (CATEGORY_PLACEMENT, CATEGORY_PLACEMENT),
        (CATEGORY_OTHER, CATEGORY_OTHER),
    ],
    description='Category',
    layout=widgets.Layout(width='50%'),
)
btn_yes = widgets.Button(description='Yes', button_style='success')
btn_no = widgets.Button(description='No', button_style='danger')
btn_unclear = widgets.Button(description='Unclear', button_style='warning')
btn_skip = widgets.Button(description='Skip')
buttons = widgets.HBox([btn_yes, btn_no, btn_unclear, btn_skip])


def _show_current():
    img_out.clear_output(wait=True)
    if state['idx'] >= len(queue):
        status_label.value = (
            f"<b>Session complete.</b> Labeled {state['done']} scene(s) this run."
        )
        meta_label.value = ''
        note_box.value = ''
        return
    item = queue[state['idx']]
    remaining = len(queue) - state['idx']
    status_label.value = (
        f"Scene {state['idx'] + 1} / {len(queue)} "
        f"(remaining in batch: {remaining}; labeled this run: {state['done']})"
    )
    meta_label.value = (
        f"<b>ep {item['episode_index']}</b> "
        f"[{item['category']}] term=<code>{item['spatial_term']}</code><br>"
        f"A: {item['instruction']}<br>"
        f"B: {item['instr_b']}<br>"
        f"current feasible_both=<code>{item['feasible_both']}</code>"
    )
    note_box.value = item.get('feasibility_note', '') or ''
    cat_dd.value = item.get('category_manual', '') or ''
    with img_out:
        paths = [item['image_path']]
        titles = ['initial']
        if item.get('grasp_image_path'):
            paths.append(item['grasp_image_path'])
            titles.append('grasp')
        fig, axes = plt.subplots(1, len(paths), figsize=(6 * len(paths), 5))
        if len(paths) == 1:
            axes = [axes]
        for ax, rel, title in zip(axes, paths, titles):
            img = Image.open(os.path.join(CACHE_DIR, rel))
            ax.imshow(img)
            ax.set_title(title)
            ax.axis('off')
        plt.tight_layout()
        plt.show()


def _save(feasible: str):
    if state['idx'] >= len(queue):
        return
    item = queue[state['idx']]
    ep = int(item['episode_index'])
    note = {'feasible_both': feasible}
    text = note_box.value.strip()
    if text:
        note['feasibility_note'] = text
    if cat_dd.value:
        note['category_manual'] = cat_dd.value
    update_manifest_annotations(CACHE_DIR, {ep: note})
    item['feasible_both'] = feasible
    state['done'] += 1
    state['idx'] += 1
    _show_current()


def _skip(_):
    if state['idx'] >= len(queue):
        return
    state['idx'] += 1
    _show_current()


btn_yes.on_click(lambda _: _save('yes'))
btn_no.on_click(lambda _: _save('no'))
btn_unclear.on_click(lambda _: _save('unclear'))
btn_skip.on_click(_skip)

ui = widgets.VBox([status_label, meta_label, img_out, note_box, cat_dd, buttons])
display(ui)
_show_current()

## 5. Progress after the session

Re-print the summary so the session's writes are visible without reloading
the notebook.

In [ ]:
summary = review_summary(CACHE_DIR)
print(f"total scenes: {summary['total']}")
print(f"pairable: {summary['pairable']}  non-pairable: {summary['non_pairable']}")
print(
    f"referent_selection pairable: "
    f"unreviewed={summary['referent_pairable_unreviewed']}  "
    f"yes={summary['referent_pairable_yes']}"
)
print('\nby category x feasible_both:')
for (cat, feas), n in sorted(summary['by_category_feasibility'].items()):
    print(f'  {cat:22} {feas:12} {n}')